# Weapon Detection Fine-Tuning (YOLOv8s) — OD-WeaponDetection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mukesh-dev-git/mukesh-fs-ai-project/blob/main/notebooks/02_weapon_detection.ipynb)

This notebook implements **Task 3 / Phase 1b** of the Crime Scene Analysis Project:
1. Ingests the **OD-WeaponDetection** dataset (pistol and knife detection subsets).
2. Converts Pascal VOC XML annotations into normalized YOLO format (`[class, x_center, y_center, width, height]`).
3. Constructs a leak-free 80/20 train/validation split and generates `data.yaml`.
4. Fine-tunes a pretrained **YOLOv8s** detector on weapon classes (`pistol`, `knife`).
5. Evaluates and reports **mAP@50**, **mAP@50–95**, per-class precision, recall, and confusion matrix.
6. Exports `yolov8s_weapon_best.pt` and wires it directly into **Stage 3 (Evidence Detection)** of the main crime scene analysis pipeline.


## 0. Setup and Environment


In [ ]:
!pip install -q ultralytics kagglehub opencv-python pillow pyyaml matplotlib scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os, re, sys, glob, shutil, random, xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import yaml
import torch
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch Device:", device)


## 1. Dataset Ingestion (OD-WeaponDetection)

The dataset contains firearms (`pistol`) and bladed weapons (`knife`) annotated in Pascal VOC XML format.
We download it via `kagglehub` using your Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`), with fallback to GitHub clone.


In [ ]:
from google.colab import userdata
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Loaded Kaggle credentials from Colab Secrets.")
except Exception as e:
    print(f"Notice: Colab secrets not found ({e}). Ensure KAGGLE_USERNAME and KAGGLE_KEY are set.")

import kagglehub

DATA_ROOT = None
try:
    path = kagglehub.dataset_download("ari-dasci/od-weapondetection")
    DATA_ROOT = Path(path)
    print("Successfully downloaded via Kagglehub:", DATA_ROOT)
except Exception as e:
    print(f"Kaggle download failed ({e}). Cloning directly from GitHub...")
    git_dir = Path("/content/OD-WeaponDetection")
    if not git_dir.exists():
        !git clone --depth 1 https://github.com/ari-dasci/OD-WeaponDetection.git /content/OD-WeaponDetection
    DATA_ROOT = git_dir
    print("Using GitHub repo path:", DATA_ROOT)

print("Contents of DATA_ROOT:")
for p in sorted(DATA_ROOT.glob("*"))[:10]:
    print(" ", p.name)


## 2. Convert Pascal VOC (XML) to YOLO Format

Pascal VOC annotations specify bounding boxes as absolute pixel coordinates:
$$\text{bbox} = [x_{\min}, y_{\min}, x_{\max}, y_{\max}]$$

YOLO format requires normalized center coordinates and dimensions relative to image width $W$ and height $H$:
$$x_{\text{center}} = \frac{x_{\min} + x_{\max}}{2 \times W}, \quad y_{\text{center}} = \frac{y_{\min} + y_{\max}}{2 \times H}$$
$$w = \frac{x_{\max} - x_{\min}}{W}, \quad h = \frac{y_{\max} - y_{\min}}{H}$$

Target Classes:
- `0`: `pistol`
- `1`: `knife`


In [ ]:
CLASSES = ["pistol", "knife"]
CLASS_MAP = {
    "pistol": 0, "gun": 0, "handgun": 0, "firearm": 0,
    "knife": 1, "dagger": 1, "blade": 1
}

def voc_to_yolo(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    boxes = []
    
    # Check image size in XML if available
    size_elem = root.find("size")
    if size_elem is not None:
        w_val = int(size_elem.find("width").text or img_w)
        h_val = int(size_elem.find("height").text or img_h)
        if w_val > 0 and h_val > 0:
            img_w, img_h = w_val, h_val

    for obj in root.findall("object"):
        name = obj.find("name").text.lower().strip()
        if name not in CLASS_MAP:
            continue
        cls_id = CLASS_MAP[name]
        bnd = obj.find("bndbox")
        xmin = float(bnd.find("xmin").text)
        ymin = float(bnd.find("ymin").text)
        xmax = float(bnd.find("xmax").text)
        ymax = float(bnd.find("ymax").text)
        
        # Clip to image bounds
        xmin = max(0.0, min(xmin, img_w))
        xmax = max(0.0, min(xmax, img_w))
        ymin = max(0.0, min(ymin, img_h))
        ymax = max(0.0, min(ymax, img_h))
        
        if xmax <= xmin or ymax <= ymin:
            continue
            
        xc = ((xmin + xmax) / 2.0) / img_w
        yc = ((ymin + ymax) / 2.0) / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h
        boxes.append((cls_id, xc, yc, bw, bh))
    return boxes

# Discover XML annotations and matching image pairs
xml_files = list(DATA_ROOT.rglob("*.xml"))
print(f"Discovered {len(xml_files)} XML annotation files.")

pairs = []
img_exts = [".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".PNG"]
for xf in xml_files:
    img_match = None
    for ext in img_exts:
        candidate = xf.with_suffix(ext)
        if candidate.exists():
            img_match = candidate
            break
        # Also check same name in sibling 'images' or parent folders
        parent_candidate = xf.parent.parent / "images" / (xf.stem + ext)
        if parent_candidate.exists():
            img_match = parent_candidate
            break
    if img_match:
        pairs.append((img_match, xf))

print(f"Matched {len(pairs)} image-annotation pairs.")


## 3. Construct 80/20 Train/Validation Split and Directory Tree

We partition the matched image-label pairs into an 80% training set and a 20% validation set,
then copy/format the data into standard Ultralytics YOLO structure.


In [ ]:
DEST = Path("/content/yolo_weapon_dataset")
for split in ["train", "val"]:
    (DEST / split / "images").mkdir(parents=True, exist_ok=True)
    (DEST / split / "labels").mkdir(parents=True, exist_ok=True)

random.seed(42)
random.shuffle(pairs)

split_idx = int(0.8 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

print(f"Splits: {len(train_pairs)} train pairs, {len(val_pairs)} val pairs.")

def process_pairs(pair_list, split_name):
    written = 0
    for img_path, xml_path in pair_list:
        try:
            im = Image.open(img_path)
            w, h = im.size
            boxes = voc_to_yolo(xml_path, w, h)
            if not boxes:
                continue
                
            dest_img = DEST / split_name / "images" / img_path.name
            dest_lbl = DEST / split_name / "labels" / (img_path.stem + ".txt")
            
            shutil.copy2(img_path, dest_img)
            with open(dest_lbl, "w") as f:
                for b in boxes:
                    f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")
            written += 1
        except Exception as e:
            continue
    return written

n_train = process_pairs(train_pairs, "train")
n_val = process_pairs(val_pairs, "val")
print(f"Successfully processed: {n_train} train images, {n_val} val images with valid labels.")


## 4. Generate `data.yaml` Specification

Ultralytics YOLO requires a dataset YAML definition file pointing to the training and validation image sets.


In [ ]:
yaml_content = {
    "path": str(DEST.resolve()),
    "train": "train/images",
    "val": "val/images",
    "names": {0: "pistol", 1: "knife"}
}

yaml_path = DEST / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"Created {yaml_path}:")
print(yaml_path.read_text())


## 5. Fine-Tune YOLOv8s on Weapon Detection

We initialize weights from `yolov8s.pt` (pretrained on COCO) and fine-tune on the pistol and knife dataset.
Using `imgsz=640` and batch size 16 on the T4 GPU provides fast convergence.


In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8s backbone
model = YOLO("yolov8s.pt")

EPOCHS = 15
BATCH_SIZE = 16

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else "cpu",
    project="weapon_experiments",
    name="yolov8s_weapons",
    save=True,
    plots=True,
    verbose=True
)


## 6. Model Evaluation (mAP@50, mAP@50–95 & Confusion Matrix)

We compute validation metrics to evaluate weapon detection performance across both classes.


In [ ]:
best_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
print("Best model checkpoint:", best_weights)

# Evaluate best checkpoint
fine_tuned = YOLO(str(best_weights))
metrics = fine_tuned.val(data=str(yaml_path))

print("\n" + "="*50)
print("=== WEAPON DETECTION EVALUATION METRICS ===")
print("="*50)
print(f"Overall mAP@50:     {metrics.box.map50:.4f}")
print(f"Overall mAP@50-95:  {metrics.box.map:.4f}")

for i, cls_name in enumerate(CLASSES):
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
    print(f"  Class [{cls_name}]: Precision={p:.4f} | Recall={r:.4f} | mAP@50={ap50:.4f}")

# Display training & validation summary plots if available
plot_dir = Path(model.trainer.save_dir)
for plot_name in ["confusion_matrix.png", "results.png"]:
    p = plot_dir / plot_name
    if p.exists():
        im = Image.open(p)
        plt.figure(figsize=(10, 6))
        plt.imshow(im)
        plt.axis("off")
        plt.title(plot_name)
        plt.show()


## 7. Qualitative Visualizations on Test Imagery

We run inference on validation images to visualize bounding boxes and confidence scores for detected weapons.


In [ ]:
val_imgs = list((DEST / "val" / "images").glob("*"))[:6]

plt.figure(figsize=(15, 10))
for idx, img_p in enumerate(val_imgs):
    res = fine_tuned(str(img_p), conf=0.3, verbose=False)[0]
    res_bgr = res.plot()
    res_rgb = cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 3, idx + 1)
    plt.imshow(res_rgb)
    plt.title(f"Detections: {len(res.boxes)}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 8. Wire Fine-Tuned Weights into Stage 3 of Crime Scene Pipeline

In the main crime scene analysis pipeline (`01_scene_analysis_pipeline.ipynb`), Stage 3 uses YOLO to detect evidence objects.
Here we copy the fine-tuned model checkpoint to `yolov8s_weapon_best.pt` and demonstrate how Stage 3 loads it to detect both weapons and generic scene evidence.


In [ ]:
# Save best model to current working directory
export_path = Path("yolov8s_weapon_best.pt")
shutil.copy2(best_weights, export_path)
print(f"Exported model checkpoint to: {export_path.resolve()} ({export_path.stat().st_size / 1e6:.1f} MB)")

# Define the unified Stage 3 evidence detector
coco_detector = YOLO("yolov8s.pt")
weapon_detector = YOLO("yolov8s_weapon_best.pt")

COCO_EVIDENCE = {"bottle", "cell phone", "handbag", "backpack", "suitcase", "car", "motorcycle", "truck", "bicycle"}

def detect_forensic_evidence(img_bgr, conf=0.25):
    evidence = []
    # 1. Specialized weapon detection (pistol, knife)
    w_res = weapon_detector(img_bgr, conf=conf, verbose=False)[0]
    for b in w_res.boxes:
        label = w_res.names[int(b.cls)]
        evidence.append({
            "label": label,
            "conf": round(float(b.conf), 2),
            "box": [round(v) for v in b.xyxy[0].tolist()],
            "source": "fine-tuned-weapon-model"
        })
    # 2. General evidence objects from COCO
    c_res = coco_detector(img_bgr, conf=conf, verbose=False)[0]
    for b in c_res.boxes:
        label = c_res.names[int(b.cls)]
        if label in COCO_EVIDENCE:
            evidence.append({
                "label": label,
                "conf": round(float(b.conf), 2),
                "box": [round(v) for v in b.xyxy[0].tolist()],
                "source": "coco-detector"
            })
    return evidence

# Test on a validation image
test_img = cv2.imread(str(val_imgs[0]))
detected = detect_forensic_evidence(test_img)
print("Unified Stage 3 Evidence Detections on test image:")
print(json.dumps(detected, indent=2))


## 9. Conclusion and Next Steps

- **Checkpoint:** Saved `yolov8s_weapon_best.pt`.
- **Pipeline Wiring:** Stage 3 now supports specialized weapon classes (`pistol`, `knife`) alongside generic COCO evidence classes.
- **Next Task:** Task 4 — Build the 4th model (Temporal Video Anomaly Model on UCF-Crime clips).
